# SQL Dataset Merge + Cleaning

Bu notebook, iki farkli SQL dataset'ini ortak formata getirir, birlestirir,
EDA ile uyumlu kolon temizligi ve etiket normalizasyonu uygular: `true` / `yes` / `positive` /
`malicious` gibi degerler 1'e, `false` / `no` / `negative` / `benign` gibi degerler 0'a map'lenir;
bilinmeyen etiketler atilir; `(Sentence, Label)` yinelenenleri silinir. Sonunda yeni bir CSV dosyasi olusturur.

In [ ]:
import math
import os

import pandas as pd

## 1) Veri Kaynaklarini Yukle, Karistir ve Birlestir

- Her iki CSV `encoding="UTF-8"` ile okunur (EDA ile ayni).
- `looking for beter dataset/dataset.csv` dosyasindan sadece `full_query` ve `label` alinir.
- Kolon adlari `Sentence` ve `Label` olarak degistirilir.
- Ilk dataset tamamen rastgele karistirilir ve indeks sifirlanir (`random_state=42`).
- `SQLiV3.csv`: EDA'daki gibi `Unnamed: 2` ve `Unnamed: 3` kolonlari kaldirilir, ardindan `Sentence` ve `Label` secilir.
- Iki dataset birlestirilir.
- Birlestirilen yeni dataset son kez tamamen rastgele karistirilir ve indeks yeniden sifirlanir (`random_state=42`).

In [ ]:
PATH_BETTER = "./data/looking for beter dataset/dataset.csv"  # Ilk dataset dosya yolu
PATH_SQLIV3 = "./data/SQLiV3.csv"  # Ikinci dataset dosya yolu
OUTPUT_PATH = "./data/merged_cleaned_preprocessed.csv"  # Cikti dosya yolu

# 1) Ilk dataseti oku (EDA ile ayni encoding)
better_df = pd.read_csv(PATH_BETTER, low_memory=False, encoding="UTF-8")
better_df = better_df[["full_query", "label"]].copy()
better_df = better_df.rename(columns={"full_query": "Sentence", "label": "Label"})
better_df = better_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2) SQLiV3: Dataset_EDA.ipynb ile ayni — once Unnamed kolonlari, sonra Sentence/Label
sqliv3_df = pd.read_csv(PATH_SQLIV3, encoding="UTF-8")
sqliv3_df = sqliv3_df.drop(columns=["Unnamed: 2", "Unnamed: 3"], errors="ignore")

if "Sentence" not in sqliv3_df.columns or "Label" not in sqliv3_df.columns:
    raise ValueError("SQLiV3.csv dosyasinda 'Sentence' ve 'Label' kolonlari bulunamadi.")

sqliv3_df = sqliv3_df[["Sentence", "Label"]].copy()

# 3) Iki dataseti birlestir ve karistir
merged_df = pd.concat([sqliv3_df, better_df], ignore_index=True)
merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("SQLiV3 shape:", sqliv3_df.shape)
print("Better dataset (first shuffle) shape:", better_df.shape)
print("Merged dataset (final shuffle) shape:", merged_df.shape)
merged_df.head()

C:\Users\Mrt\AppData\Local\Temp\ipykernel_13504\4262484001.py:6: DtypeWarning: Columns (0: attack_stage, 1: tamper_method, 2: attack_status, 3: attack_id, 4: attack_technique) have mixed types. Specify dtype option on import or set low_memory=False.
  better_df = pd.read_csv(PATH_BETTER)


SQLiV3 shape: (30919, 2)
Better dataset shape: (3688977, 2)
Merged shape: (3719896, 2)


,Sentence,Label
0,""" or pg_sleep ( __TIME__ ) --",1
1,create user name identified by pass123 tempora...,NaN
2,AND 1 = utl_inaddr.get_host_address ( ...,1
3,select * from users where id = '1' or @ @1 ...,1
4,"select * from users where id = 1 or 1#"" ( ...",1


## 2) Data Cleaning

- `Label` icin baslangicta NaN satirlar kaldirilir.
- Etiketler 0/1 tamsayiya **map** edilir: `0`/`1`, `true`/`yes`, `false`/`no`, `positive`/`negative`,
  `pos`/`neg`, `+1`/`-1`, `sqli`/`malicious`/`attack`, `benign`/`normal`/`safe` ve benzeri sinif adlari
  (kucuk harf duyarsiz); sayisal `1.0`/`0.0` stringleri de kabul edilir. Map edilemeyen etiketler silinir.
- `Sentence` metni degistirilmez (EDA ile ayni).
- Ayni `(Sentence, Label)` tekrarlari `drop_duplicates` ile atilir.

In [ ]:
def map_label_to_binary(raw):
    """true/yes/positive/sqli -> 1; false/no/negative/benign -> 0; taninmayan -> NaN."""
    if pd.isna(raw):
        return float("nan")
    if isinstance(raw, bool):
        return int(raw)
    if isinstance(raw, int):
        if raw == 1:
            return 1
        if raw == 0:
            return 0
        return float("nan")
    if isinstance(raw, float):
        if math.isnan(raw):
            return float("nan")
        if raw == 1.0:
            return 1
        if raw == 0.0:
            return 0
        return float("nan")
    s = str(raw).strip().lower()
    if s in {
        "1",
        "1.0",
        "+1",
        "true",
        "yes",
        "t",
        "y",
        "positive",
        "pos",
        "sqli",
        "sql_injection",
        "malicious",
        "attack",
        "harmful",
        "unsafe",
        "bad",
        "suspicious",
        "anomaly",
        "anomalous",
        "class_1",
        "label_1",
        "class1",
        "label1",
        "malware",
        "exploit",
    }:
        return 1
    if s in {
        "0",
        "0.0",
        "-1",
        "false",
        "no",
        "f",
        "n",
        "negative",
        "neg",
        "benign",
        "normal",
        "safe",
        "clean",
        "legitimate",
        "harmless",
        "good",
        "non-malicious",
        "non_malicious",
        "nonmalicious",
        "class_0",
        "label_0",
        "class0",
        "label0",
        "innocuous",
        "valid",
        "acceptable",
    }:
        return 0
    try:
        v = float(s)
        if v == 1.0:
            return 1
        if v == 0.0:
            return 0
    except ValueError:
        pass
    return float("nan")


clean_df = merged_df.copy()
clean_df = clean_df.dropna(subset=["Label"]).copy()
clean_df["Label"] = clean_df["Label"].apply(map_label_to_binary)
clean_df = clean_df.dropna(subset=["Label"]).copy()
clean_df["Label"] = clean_df["Label"].astype(int)
clean_df = clean_df.drop_duplicates(subset=["Sentence", "Label"]).reset_index(drop=True)

print("Cleaned shape:", clean_df.shape)
print(clean_df["Label"].value_counts(dropna=False))
clean_df.head()

Cleaned + preprocessed shape: (2322915, 2)
Label
0    2001191
1     321724
Name: count, dtype: int64


,Sentence,Label
0,""" or pg_sleep ( __time__ ) --",1
1,and 1 = utl_inaddr.get_host_address ( ( select...,1
2,select * from users where id = '1' or @ @1 = 1...,1
3,"select * from users where id = 1 or 1#"" ( unio...",1
4,select name from syscolumns where id = ( selec...,1


## 3) Son dataseti karistir ve CSV kaydet

Tum temizlikten sonra satirlar tekrar rastgele siralanir (`sample`, `random_state=42`) — etiketler
homojen dagilsin ve train/val split oncesi bloklanma azalsin diye. Ardından UTF-8 CSV yazilir.

In [ ]:
# Cikti klasorunu garanti et
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

final_df = clean_df[["Sentence", "Label"]].copy()
# Son adim: tum islemlerden sonra homojen siralama (onceki shuffle ile ayni tohum)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Yeni CSV olusturuldu:")
print(OUTPUT_PATH)
print("Satir sayisi:", len(final_df))
final_df.head(5)

Yeni CSV olusturuldu:
./data/merged_cleaned_preprocessed.csv
Satir sayisi: 2322915


,Sentence,Label
0,""" or pg_sleep ( __time__ ) --",1
1,and 1 = utl_inaddr.get_host_address ( ( select...,1
2,select * from users where id = '1' or @ @1 = 1...,1
3,"select * from users where id = 1 or 1#"" ( unio...",1
4,select name from syscolumns where id = ( selec...,1


In [ ]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2322915 entries, 0 to 2322914
Data columns (total 2 columns):
 #   Column    Dtype
---  ------    -----
 0   Sentence  str  
 1   Label     int64
dtypes: int64(1), str(1)
memory usage: 35.4 MB
